In [1]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBClassifier
from scipy.stats import uniform
import pandas as pd

In [ ]:
data = pd.read_csv('datasets/agmp/gyro_mobile.csv')
data.drop(columns=['timestamp'], inplace=True)

x = data[data.columns[:6]]
y = data[data.columns[6:]]

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=0, stratify=y['Activity'])

classratio = len(data[data['Activity']==0]) / len(data[data['Activity']==1])

param_dist = {
    'max_depth': [1,2,3,4,5,6,7,8,9,10],
    'n_estimators': [100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1400,1500],
    'min_child_weight': [1,2,3,4,5,6,7,8,9,10],
    'subsample': uniform(0.4, 0.6),
    'colsample_bytree': uniform(0.1, 0.9),
    'learning_rate': uniform(0.01, 0.29)
}

xgb = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    scale_pos_weight=classratio,
    n_jobs=-1,
)

random_search = RandomizedSearchCV(
    estimator=xgb, 
    param_distributions=param_dist, 
    scoring='balanced_accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
random_search.fit(xtrain, ytrain)

# Print best parameters
print(f"Best parameters: {random_search.best_params_}")
print(f"Best score: {random_search.best_score_}")

Fitting 5 folds for each of 250 candidates, totalling 1250 fits


/home/denis/arduino-xgboost/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
675 fits failed out of a total of 1250.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/home/denis/arduino-xgboost/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/denis/arduino-xgboost/.venv/lib/python3.10/site-packages/xgboost/core.py", line 726, in inner_f
    return func(**kwargs)
  File "/home/denis/arduino-xgboost/.venv/lib/python3.10/site-packages/xgboost/sklearn.py", line 1599, in fit
    self._Booster = tr

Best parameters: {'colsample_bytree': np.float64(0.6510791485078967), 'learning_rate': np.float64(0.04957097492211378), 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 100, 'subsample': np.float64(0.5974933842150852)}
Best score: 0.8861330429350627


In [40]:
RSresults = random_search.cv_results_
RSresultsdf = pd.DataFrame(RSresults)
RSresultsdf.to_csv('agmp_randomSearchResults1.csv', index=False)

In [36]:
top10results = RSresultsdf[(RSresultsdf['rank_test_score'] >= 1) & (RSresultsdf['rank_test_score'] <= 10)]

minCST = top10results['param_colsample_bytree'].min()
maxCST = top10results['param_colsample_bytree'].max()
minLR = top10results['param_learning_rate'].min()
maxLR = top10results['param_learning_rate'].max()
minMCW = top10results['param_min_child_weight'].min()
maxMCW = top10results['param_min_child_weight'].max()
minMD = top10results['param_max_depth'].min()
maxMD = top10results['param_max_depth'].max()
minNS = top10results['param_n_estimators'].min()
maxNS = top10results['param_n_estimators'].max()
minSS = top10results['param_subsample'].min()
maxSS = top10results['param_subsample'].max()

print(f"minCST: {minCST}, maxCST: {maxCST}")
print(f"minLR: {minLR}, maxLR: {maxLR}")
print(f"minMCW: {minMCW}, maxMCW: {maxMCW}")
print(f"minMaxDepth: {minMD}, maxMaxDepth: {maxMD}")
print(f"minNestimators: {minNS}, maxNestimators: {maxNS}")
print(f"minSS: {minSS}, maxSS: {maxSS}")

minCST: 0.3444255920016027, maxCST: 0.8980468339125637
minLR: 0.020548017914858253, maxLR: 0.09244707096027897
minMCW: 3, maxMCW: 9
minMaxDepth: 2, maxMaxDepth: 10
minNestimators: 100, maxNestimators: 1100
minSS: 0.5355818023100279, maxSS: 0.9834086165987527


In [ ]:
nmd = range(minMD, maxMD+1)
nns = range(minNS, maxNS+1)
nmcw = range(minMCW, maxMCW+1)

param_dist2 = {
    'max_depth': nmd,
    'n_estimators': nns,
    'min_child_weight': nmcw,
    'subsample': uniform(minSS, (maxSS-minSS)),
    'colsample_bytree': uniform(minCST, (maxCST-minCST)),
    'learning_rate': uniform(minLR, (maxLR-minLR))
}

random_search2 = RandomizedSearchCV(
    estimator=xgb, 
    param_distributions=param_dist2, 
    scoring='balanced_accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
random_search2.fit(xtrain, ytrain)

# Print best parameters
print(f"Best parameters: {random_search2.best_params_}")
print(f"Best score: {random_search2.best_score_}")


Fitting 5 folds for each of 250 candidates, totalling 1250 fits
Best parameters: {'colsample_bytree': np.float64(0.5106168525562308), 'learning_rate': np.float64(0.03972983384203177), 'max_depth': 9, 'min_child_weight': 4, 'n_estimators': 126, 'subsample': np.float64(0.6901437508034196)}
Best score: 0.8880493146063053


In [41]:

bestScores2 = pd.DataFrame(random_search2.cv_results_)
bestScores2.to_csv('agmp_randomSearchResults2.csv', index=False)

- [How to Configure the Gradient Boosting Algorithm](https://machinelearningmastery.com/configure-gradient-boosting-algorithm/)
- [Hyperparameter Tuning XGBoost with early stopping](https://macalusojeff.github.io/post/HyperparameterTuningXGB/)



In [39]:
xval, xtestnew, yval, ynew = train_test_split(xtest, ytest, test_size=0.5, random_state=0, stratify=ytest['Activity'])

xgb2 = XGBClassifier(
    objective='binary:logistic',
    tree_method='exact',
    scale_pos_weight=classratio,
    n_jobs=-1,
    early_stopping_rounds=10
)
param_dist2 = {
    'max_depth': nmd,
    'n_estimators': nns,
    'min_child_weight': nmcw,
    'subsample': uniform(minSS, (maxSS-minSS)),
    'colsample_bytree': uniform(minCST, (maxCST-minCST)),
    'learning_rate': uniform(minLR, (maxLR-minLR))
}
random_search3 = RandomizedSearchCV(
    estimator=xgb, 
    param_distributions=param_dist2, 
    scoring='balanced_accuracy',
    n_iter=50, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
random_search3.fit(xtrain, ytrain, eval_set=[(xval, yval)])

# Print best parameters
print(f"Best parameters: {random_search3.best_params_}")
print(f"Best score: {random_search3.best_score_}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[0]	validation_0-logloss:0.66478
[0]	validation_0-logloss:0.66642
[0]	validation_0-logloss:0.67063
[0]	validation_0-logloss:0.66921
[0]	validation_0-logloss:0.66858
[0]	validation_0-logloss:0.67299
[1]	validation_0-logloss:0.62974
[0]	validation_0-logloss:0.67246
[1]	validation_0-logloss:0.64238
[0]	validation_0-logloss:0.67421
[1]	validation_0-logloss:0.63064
[1]	validation_0-logloss:0.64162
[0]	validation_0-logloss:0.67181
[1]	validation_0-logloss:0.63880
[0]	validation_0-logloss:0.67201
[0]	validation_0-logloss:0.66491
[0]	validation_0-logloss:0.66591
[1]	validation_0-logloss:0.64738
[2]	validation_0-logloss:0.60709
[1]	validation_0-logloss:0.64761
[2]	validation_0-logloss:0.63280
[1]	validation_0-logloss:0.62779
[0]	validation_0-logloss:0.66760
[0]	validation_0-logloss:0.66398
[2]	validation_0-logloss:0.60774
[1]	validation_0-logloss:0.65431[1]	validation_0-logloss:0.62840

[2]	validation_0-logloss:0.63222
[1]	validation